In [ ]:
import json
import sys
import time

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML

import websocket

# ==============================================================================
# Import Protobuf Decoder
# ==============================================================================

PROTO_PATH = Path(
    r"D:\Data Projects\MEXC API Architecture\websocket-proto"
)

sys.path.append(str(PROTO_PATH))

from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# ==============================================================================
# WebSocket Configuration
# ==============================================================================

WS_URL = "wss://wbs-api.mexc.com/ws"

SYMBOL = "ETHUSDT"

ENDPOINT = f"spot@public.aggre.depth.v3.api.pb@10ms@{SYMBOL}"

subscription = {
    "method": "SUBSCRIPTION",
    "params": [
        ENDPOINT
    ],
    "id": 1
}

received_time = lambda: datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S.%f UTC"
)

# ==============================================================================
# Connect
# ==============================================================================

ws = websocket.create_connection(
    WS_URL
)

ws.send(
    json.dumps(subscription)
)

# ==============================================================================
# Dashboard Refresh Control
# ==============================================================================
last_display_update = 0
DISPLAY_REFRESH_RATE = 2.0   # seconds

dashboard_display = display(
    HTML(""),
    display_id=True
)

chart_display = display(
    HTML(""),
    display_id=True
)

# ==============================================================================
# Live Order Book Liquidity Chart
# ==============================================================================
import matplotlib.pyplot as plt
from collections import deque
import math

# ==============================================================================
# Chart Storage
# ==============================================================================
LIQUIDITY_HISTORY_SIZE = 200

strongest_bid_wall_history = deque(maxlen=LIQUIDITY_HISTORY_SIZE)

strongest_ask_wall_history = deque(maxlen=LIQUIDITY_HISTORY_SIZE)

# ==============================================================================
# Create Scatter Chart
# ==============================================================================
fig, ax = plt.subplots(
    figsize=(14, 4)
)

bid_scatter = ax.scatter(
    [],
    [],
    color="green",
    s=40,
    label="Strongest Bid Wall"
)

ask_scatter = ax.scatter(
    [],
    [],
    color="red",
    s=40,
    label="Strongest Ask Wall"
)

ax.set_title("Strongest Order Book Walls")

ax.set_xlabel("WebSocket Batch")

ax.set_ylabel("USDT")

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0)
)

ax.grid(
    True,
    alpha=0.3
)

# --------------------------------------------------------------------------
# Initial Axis Range
# --------------------------------------------------------------------------
ax.set_ylim(
    0,
    1
)

ax.set_xlim(
    0,
    LIQUIDITY_HISTORY_SIZE
)

plt.tight_layout()

# ==============================================================================
# Chart Update Function
# ==============================================================================

def update_liquidity_chart(
    strongest_bid_wall,
    strongest_ask_wall
):

    # --------------------------------------------------------------------------
    # Handle Missing Order Book Sides
    # --------------------------------------------------------------------------
    if strongest_bid_wall is None:
        strongest_bid_wall = float("nan")

    if strongest_ask_wall is None:
        strongest_ask_wall = float("nan")

    # --------------------------------------------------------------------------
    # Store History
    # --------------------------------------------------------------------------
    strongest_bid_wall_history.append(
        strongest_bid_wall
    )

    strongest_ask_wall_history.append(
        strongest_ask_wall
    )

    # --------------------------------------------------------------------------
    # Prepare Scatter Data
    # --------------------------------------------------------------------------
    bid_points = [
        (i, value)
        for i, value in enumerate(strongest_bid_wall_history)
        if value == value
    ]

    ask_points = [
        (i, value)
        for i, value in enumerate(strongest_ask_wall_history)
        if value == value
    ]

    # --------------------------------------------------------------------------
    # Update Scatter Objects
    # --------------------------------------------------------------------------

    if bid_points:
        bid_scatter.set_offsets(bid_points)
    else:
        bid_scatter.set_offsets(
            [[float("nan"), float("nan")]]
        )

    if ask_points:
        ask_scatter.set_offsets(ask_points)
    else:
        ask_scatter.set_offsets(
            [[float("nan"), float("nan")]]
        )

    # --------------------------------------------------------------------------
    # Dynamic Y Axis
    # --------------------------------------------------------------------------
    valid_values = (
        [value for _, value in bid_points] +
        [value for _, value in ask_points]
    )

    if valid_values:
        chart_max = max(
            valid_values
        )

        current_max = ax.get_ylim()[1]

        if chart_max > current_max:
            ax.set_ylim(
                0,
                chart_max * 1.05
            )

    # --------------------------------------------------------------------------
    # X Axis Window
    # --------------------------------------------------------------------------
    ax.set_xlim(
        0,
        LIQUIDITY_HISTORY_SIZE
    )

    fig.canvas.draw_idle()

# ==============================================================================
# Wall Pressure Chart (Net Buy/Sell Pressure)
# ==============================================================================

WALL_HISTORY_SIZE = 200

wall_pressure_history = deque(
    maxlen=WALL_HISTORY_SIZE
)

# ==============================================================================
# Create Chart
# ==============================================================================

fig_pressure, ax_pressure = plt.subplots(figsize=(14,4))

pressure_bars = ax_pressure.bar(
    [],
    []
)

ax_pressure.axhline(
    0,
    color="black",
    linewidth=1
)

ax_pressure.set_title("Order Book Wall Pressure")

ax_pressure.set_xlabel("WebSocket Update")

ax_pressure.set_ylabel("Buy  ← Pressure →  Sell")

ax_pressure.grid(
    True,
    alpha=0.3
)

ax_pressure.set_xlim(
    0,
    WALL_HISTORY_SIZE
)

ax_pressure.set_ylim(
    -1,
    1
)

plt.tight_layout()

# ==============================================================================
# Update Chart
# ==============================================================================

def update_wall_pressure_chart(
    largest_bid_wall,
    largest_ask_wall
):

    pressure = 0.0

    if (
        largest_bid_wall
        and largest_ask_wall
        and largest_bid_wall["value"] > 0
        and largest_ask_wall["value"] > 0
    ):

        bid_value = largest_bid_wall["value"]
        ask_value = largest_ask_wall["value"]

        pressure = (
            bid_value - ask_value
        ) / (
            bid_value + ask_value
        )

    wall_pressure_history.append(pressure)

    ax_pressure.clear()

    x_values = range(len(wall_pressure_history))

    colors = [
        "green" if value > 0 else "red"
        for value in wall_pressure_history
    ]

    ax_pressure.bar(
        x_values,
        wall_pressure_history,
        color=colors
    )

    ax_pressure.axhline(
        0,
        color="black",
        linewidth=1
    )

    ax_pressure.set_title("Order Book Wall Pressure")

    ax_pressure.set_xlabel("WebSocket Update")

    ax_pressure.set_ylabel("Buy (+) / Sell (-)")

    ax_pressure.set_ylim(
        -1,
        1
    )

    ax_pressure.set_xlim(
        0,
        WALL_HISTORY_SIZE
    )

    ax_pressure.grid(
        True,
        alpha=0.3
    )

    fig_pressure.canvas.draw_idle()

# ==============================================================================
# Highest Liquidity Tracking
# ==============================================================================

highest_bid_liquidity = 0.0
highest_bid_liquidity_price = None

highest_ask_liquidity = 0.0
highest_ask_liquidity_price = None

# ==============================================================================
# Dashboard Displays
# ==============================================================================
dashboard_display = display(
    HTML(""),
    display_id=True
)

chart_display = display(
    HTML(""),
    display_id=True
)

pressure_chart_display = display(
    HTML(""),
    display_id=True
)

# ==============================================================================
# Receive Loop
# ==============================================================================
while True:

    # ==========================================================================
    # Receive Message
    # ==========================================================================
    message = ws.recv()

    if isinstance(message, str):
        continue

    # ==========================================================================
    # Decode Message
    # ==========================================================================
    wrapper = PushDataV3ApiWrapper()

    wrapper.ParseFromString(
        message
    )

    orderbook = wrapper.publicAggreDepths

    # ==========================================================================
    # Timestamps
    # ==========================================================================
    receive_timestamp = received_time()

    exchange_timestamp = datetime.fromtimestamp(
        wrapper.sendTime / 1000,
        tz=timezone.utc
    )

    # ==========================================================================
    # Extract Top Of Book
    # ==========================================================================
    best_bid_price = None
    best_bid_quantity = None

    best_ask_price = None
    best_ask_quantity = None

    for bid in orderbook.bids:

        if float(bid.quantity) > 0:
            best_bid_price = float(bid.price)
            best_bid_quantity = float(bid.quantity)

            break

    for ask in orderbook.asks:

        if float(ask.quantity) > 0:
            best_ask_price = float(ask.price)
            best_ask_quantity = float(ask.quantity)

            break

    # ==========================================================================
    # Calculate Spread
    # ==========================================================================
    spread = None

    if best_bid_price is not None and best_ask_price is not None:
        spread = best_ask_price - best_bid_price

    # ==========================================================================
    # Calculate Liquidity
    # ==========================================================================
    bid_liquidity = 0.0
    ask_liquidity = 0.0

    for bid in orderbook.bids:
        price = float(bid.price)
        quantity = float(bid.quantity)

        bid_liquidity += price * quantity

    for ask in orderbook.asks:
        price = float(ask.price)
        quantity = float(ask.quantity)

        ask_liquidity += price * quantity

    # ==============================================================================
    # Order Book Validation
    # ==============================================================================
    orderbook_valid = True
    orderbook_status = "VALID"

    if (
        best_bid_price is None
        or best_ask_price is None
        or bid_liquidity == 0
        or ask_liquidity == 0
    ):

        orderbook_valid = False

        if ask_liquidity == 0:
            orderbook_status = (
                "INVALID - Missing Ask Depth"
            )
        elif bid_liquidity == 0:
            orderbook_status = (
                "INVALID - Missing Bid Depth"
            )
        else:

            orderbook_status = (
                "INVALID - Incomplete Depth"
            )

    # ==============================================================================
    # Track Highest Liquidity
    # ==============================================================================

    if bid_liquidity > highest_bid_liquidity:

        highest_bid_liquidity = bid_liquidity
        highest_bid_liquidity_price = best_bid_price


    if ask_liquidity > highest_ask_liquidity:

        highest_ask_liquidity = ask_liquidity
        highest_ask_liquidity_price = best_ask_price

    # ==============================================================================
    # Liquidity Percentages
    # ==============================================================================

    total_liquidity = (
        bid_liquidity +
        ask_liquidity
    )

    buy_liquidity_percent = 0.0
    sell_liquidity_percent = 0.0

    if total_liquidity > 0:
        buy_liquidity_percent = (
            bid_liquidity /
            total_liquidity
        ) * 100

        sell_liquidity_percent = (
            ask_liquidity /
            total_liquidity
        ) * 100

    # ==========================================================================
    # Top-5 Liquidity
    # ==========================================================================
    top5_bid_liquidity = 0.0
    top5_ask_liquidity = 0.0

    for bid in orderbook.bids[:5]:
        price = float(bid.price)
        quantity = float(bid.quantity)

        top5_bid_liquidity += price * quantity

    for ask in orderbook.asks[:5]:
        price = float(ask.price)
        quantity = float(ask.quantity)

        top5_ask_liquidity += price * quantity

    # ==========================================================================
    # Mid Price
    # ==========================================================================
    mid_price = None

    if best_bid_price and best_ask_price:

        mid_price = (
            best_bid_price +
            best_ask_price
        ) / 2

    # ==========================================================================
    # Detect Largest Walls
    # ==========================================================================
    largest_bid_wall = None
    largest_ask_wall = None

    for bid in orderbook.bids:
        price = float(bid.price)
        quantity = float(bid.quantity)
        value = price * quantity

        if (
            largest_bid_wall is None
            or value > largest_bid_wall["value"]
        ):
            largest_bid_wall = {
                "price": price,
                "quantity": quantity,
                "value": value
            }

    for ask in orderbook.asks:
        price = float(ask.price)
        quantity = float(ask.quantity)
        value = price * quantity

        if (
            largest_ask_wall is None
            or value > largest_ask_wall["value"]
        ):
            largest_ask_wall = {
                "price": price,
                "quantity": quantity,
                "value": value
            }

    # ==========================================================================
    # Wall Metrics
    # ==========================================================================
    if (
         largest_bid_wall is not None
         and mid_price is not None
         and bid_liquidity > 0
     ):
        largest_bid_wall["distance"] = (
            mid_price -
            largest_bid_wall["price"]
        )

        largest_bid_wall["concentration"] = (
            largest_bid_wall["value"] /
            bid_liquidity
        ) * 100

    if (
         largest_ask_wall is not None
         and mid_price is not None
         and ask_liquidity > 0
     ):
        largest_ask_wall["distance"] = (
            largest_ask_wall["price"] -
            mid_price
        )

        largest_ask_wall["concentration"] = (
            largest_ask_wall["value"] /
            ask_liquidity
        ) * 100

    # ==========================================================================
    # Support / Resistance Walls
    # ==========================================================================
    nearest_support_wall = None
    nearest_resistance_wall = None

    if mid_price:
        for bid in orderbook.bids:
            price = float(bid.price)

            if price < mid_price:
                nearest_support_wall = price
                break

        for ask in orderbook.asks:
            price = float(ask.price)

            if price > mid_price:
                nearest_resistance_wall = price
                break
    # ==========================================================================
    # Top Of Book Imbalance
    # ==========================================================================
    top_book_imbalance = 0.0

    if (
        best_bid_quantity is not None
        and best_ask_quantity is not None
        and (best_bid_quantity + best_ask_quantity) > 0
    ):
        top_book_imbalance = (
            best_bid_quantity /
            (
                best_bid_quantity +
                best_ask_quantity
            )
        ) * 100

    # ==========================================================================
    # Refresh Dashboard
    # ==========================================================================
    current_time = time.time()

    if current_time - last_display_update >= DISPLAY_REFRESH_RATE:

        dashboard = []

        dashboard.append("=" * 82)
        dashboard.append("MARKET SNAPSHOT")
        dashboard.append("=" * 82)

        dashboard.append(f"Exchange Time : {exchange_timestamp}")
        dashboard.append(f"Receive Time  : {receive_timestamp}")

        # ==========================================================================
        # TOP OF BOOK
        # ==========================================================================
        dashboard.append("")
        dashboard.append("=" * 82)
        dashboard.append("TOP OF BOOK")
        dashboard.append("=" * 82)

        if best_bid_price is not None:
            dashboard.append(f"Best Bid      : {best_bid_price:.2f}")
        else:
            dashboard.append("Best Bid      : N/A")

        if best_ask_price is not None:
            dashboard.append(f"Best Ask      : {best_ask_price:.2f}")
        else:
            dashboard.append("Best Ask      : N/A")

        if spread is not None:
            dashboard.append(f"Spread        : {spread:.2f}")
        else:
            dashboard.append("Spread        : N/A")

        # ==========================================================================
        # LIQUIDITY
        # ==========================================================================
        dashboard.append("")
        dashboard.append("=" * 82)
        dashboard.append("LIQUIDITY")
        dashboard.append("=" * 82)

        dashboard.append(f"Bid Liquidity : {bid_liquidity:,.2f} USDT")
        dashboard.append(f"Ask Liquidity : {ask_liquidity:,.2f} USDT")
        dashboard.append(f"Buy Liquidity : {buy_liquidity_percent:.2f}%")
        dashboard.append(f"Sell Liquidity: {sell_liquidity_percent:.2f}%")
        dashboard.append(f"Top-5 Bid     : {top5_bid_liquidity:,.2f} USDT")
        dashboard.append(f"Top-5 Ask     : {top5_ask_liquidity:,.2f} USDT")
        dashboard.append(f"Top Book Imbalance : {top_book_imbalance:.2f}%")

        # ==========================================================================
        # WALLS
        # ==========================================================================
        dashboard.append("")
        dashboard.append("=" * 82)
        dashboard.append("WALLS")
        dashboard.append("=" * 82)

        dashboard.append("")
        dashboard.append("Largest Bid Wall")

        if largest_bid_wall:

            dashboard.append(f"Price           : {largest_bid_wall['price']:.2f}")
            dashboard.append(f"Size            : {largest_bid_wall['quantity']:.5f} ETH")
            dashboard.append(f"Value           : {largest_bid_wall['value']:,.2f} USDT")
            dashboard.append(f"Distance        : {largest_bid_wall.get('distance',0):.2f} USDT")
            dashboard.append(f"Concentration   : {largest_bid_wall.get('concentration',0):.2f}%")
        else:
            dashboard.append("N/A")

        dashboard.append("")
        dashboard.append("Largest Ask Wall")

        if largest_ask_wall:

            dashboard.append(f"Price           : {largest_ask_wall['price']:.2f}")
            dashboard.append(f"Size            : {largest_ask_wall['quantity']:.5f} ETH")
            dashboard.append(f"Value           : {largest_ask_wall['value']:,.2f} USDT")
            dashboard.append(f"Distance        : {largest_ask_wall.get('distance',0):.2f} USDT")
            dashboard.append(f"Concentration   : {largest_ask_wall.get('concentration',0):.2f}%")
        else:
            dashboard.append("N/A")

        # ==========================================================================
        # PRESSURE
        # ==========================================================================
        dashboard.append("")
        dashboard.append("=" * 82)
        dashboard.append("PRESSURE")
        dashboard.append("=" * 82)

        # --------------------------------------------------------------------------
        # Order Book Status
        # --------------------------------------------------------------------------
        dashboard.append(f"Order Book Status : {orderbook_status}")

        # --------------------------------------------------------------------------
        # Top Book Imbalance
        # --------------------------------------------------------------------------
        if orderbook_valid:
            dashboard.append(f"Top-of-Book Imbalance : {top_book_imbalance:.2f}%")
        else:
            dashboard.append("Top-of-Book Imbalance : N/A")

        dashboard.append("")

        # --------------------------------------------------------------------------
        # Strongest Bid Wall
        # --------------------------------------------------------------------------
        if largest_bid_wall:

            dashboard.append("Strongest Bid Wall:")
            dashboard.append(
                f"{largest_bid_wall['value']/1_000_000:.2f}M @ "
                f"{largest_bid_wall['price']:.2f}"
            )
        else:
            dashboard.append("Strongest Bid Wall:")
            dashboard.append("N/A")

        dashboard.append("")

        # --------------------------------------------------------------------------
        # Strongest Ask Wall
        # --------------------------------------------------------------------------
        if largest_ask_wall:
            dashboard.append("Strongest Ask Wall:")
            dashboard.append(
                f"{largest_ask_wall['value']/1_000_000:.2f}M @ "
                f"{largest_ask_wall['price']:.2f}"
            )
        else:
            dashboard.append("Strongest Ask Wall:")
            dashboard.append("N/A")

        dashboard.append("")

       # --------------------------------------------------------------------------
       # Wall Pressure Ratio
       # --------------------------------------------------------------------------

        if (
            largest_bid_wall
            and largest_ask_wall
            and largest_bid_wall["value"] > 0
            and largest_ask_wall["value"] > 0
        ):

            if largest_ask_wall["value"] > largest_bid_wall["value"]:
                wall_ratio = (
                    largest_ask_wall["value"] /
                    largest_bid_wall["value"]
                )

                dashboard.append("Wall Pressure Ratio:")
                dashboard.append(f"{wall_ratio:.2f}× Sell")
            else:
                wall_ratio = (
                    largest_bid_wall["value"] /
                    largest_ask_wall["value"]
                )
                dashboard.append("Wall Pressure Ratio:")
                dashboard.append(f"{wall_ratio:.2f}× Buy")
        else:
             dashboard.append("Wall Pressure Ratio:")
             dashboard.append("N/A")

        dashboard.append("=" * 82)

        # ==========================================================================
        # LIQUIDITY RECORDS
        # ==========================================================================

        dashboard.append("")
        dashboard.append("=" * 82)
        dashboard.append("LIQUIDITY RECORDS")
        dashboard.append("=" * 82)

        dashboard.append(
            f"Highest Bid Liquidity   : {highest_bid_liquidity:,.2f} USDT @ {highest_bid_liquidity_price:.2f}"
            if highest_bid_liquidity_price is not None
            else "Highest Bid Liquidity   : N/A"
        )

        dashboard.append(
            f"Highest Ask Liquidity   : {highest_ask_liquidity:,.2f} USDT @ {highest_ask_liquidity_price:.2f}"
            if highest_ask_liquidity_price is not None
            else "Highest Ask Liquidity   : N/A"
        )

        dashboard.append("=" * 82)

        # ==============================================================================
        # Refresh Dashboard and Charts
        # ==============================================================================
        update_liquidity_chart(
            largest_bid_wall["value"] if largest_bid_wall else None,
            largest_ask_wall["value"] if largest_ask_wall else None
        )

        update_wall_pressure_chart(
            largest_bid_wall,
            largest_ask_wall
        )

        dashboard_display.update(
            HTML(
                "<pre>"
                + "\n".join(dashboard)
                + "</pre>"
            )
        )

        chart_display.update(fig)

        pressure_chart_display.update(fig_pressure)

        last_display_update = current_time
